# Experiment 4: rule-boundary augmentation

**Motivation.** The reference model R1 got 17 of 200 dev tickets wrong. Twelve of those errors were in `priority`,
and in 9 of them the model had read the relevant value correctly but applied the rule on the wrong side. Eight were
amounts close to the ₹5,000 or ₹50,000 threshold, and one was a missed legal threat. The cleaned training set has few
tickets near these decision boundaries, so the rules are under-sampled exactly where the model fails.

**Method.** Each new example is a real cleaned training ticket with exactly one decision-relevant change:

| Type | Rows | Change |
|---|---|---|
| `amt` | 250 | The transaction amount is replaced by a value near ₹5,000 or ₹50,000: log-uniform between 0.4× and 2.4× the threshold, with 10% exactly on it. The new value is written in the ticket's original style (`Rs. 19389`, `₹46,191`, `47.5k`, `Rs 17,914.99`). |
| `legal` | 120 | A legal-threat sentence is appended. The sentences are taken from the training tickets themselves, in the same language as the ticket. |

**Safeguards.**
- `priority` and `needs_human` are recomputed with the SCHEMA.md rules, using the same rule functions that
  reproduce 100% of the cleaned labels (re-checked below).
- Each new amount is asserted to be grounded in the new text, i.e. the amount parser finds it as the claim amount.
- Every new row passes the same validation gate as the cleaned data.
- Only `train.jsonl` (through `train_clean.jsonl`) and `SCHEMA.md` are used; dev and test are never read.
- Sampling is seeded (`random.Random(1234)`), so the output is identical on every run.

**Outputs.**
- `prep_outputs/train_aug.jsonl`: the 1,262 clean rows plus 370 augmented rows, 1,632 in total.
- `prep_outputs/augmentation_log.csv`: one line per new row, with its source ticket and priority change.

A CPU runtime is enough for this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, re, math, random, collections
import pandas as pd

BASE = '/content/drive/MyDrive/vima3ya'
PREP = f'{BASE}/prep_outputs'

def load_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train = load_jsonl(f'{PREP}/train_clean.jsonl')
KEYS = ['category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human']
HDR = re.compile(r'^Received: (\d{4}-\d{2}-\d{2}) \((\w{3})\)\nSource: ([\w-]+)\n\n')

user_of, body_of, LAB = {}, {}, {}
for r in train:
    u = r['messages'][1]['content']
    m = HDR.match(u)
    assert m, r['id']
    user_of[r['id']], body_of[r['id']] = u, u[m.end():]
    LAB[r['id']] = json.loads(r['messages'][2]['content'])
keep_ids   = [r['id'] for r in train]
SYS_PROMPT = train[0]['messages'][0]['content']
print(len(train), 'clean rows loaded')          # expect 1262

Mounted at /content/drive
1262 clean rows loaded


## 1. The SCHEMA.md rules as code

These are the same functions as in the data audit (`data_prep.ipynb`). As a safety check they are re-run on every
cleaned row before being used to label new examples. Any disagreement would stop the notebook.

In [ ]:
def first_part(body):                      # the customer's own text: everything before quoted earlier emails
    return body.split('\n>')[0]

ENUMS = {
    'category': {'payment_failed', 'refund', 'fraud', 'account_access', 'kyc', 'offers', 'other'},
    'priority': {'P1', 'P2', 'P3'},
    'channel':  {None, 'upi', 'card', 'netbanking', 'wallet'},
    'language': {'en', 'hi-en'},
}
TXN_CATS = {'payment_failed', 'refund', 'fraud', 'offers'}
NON_TXN  = {'account_access', 'kyc', 'other'}
TXN_F    = ['amount_paise', 'txn_date', 'txn_id', 'channel']

def violations(y):
    v = []
    if list(y) != KEYS:
        v.append('keys')
    for k, allowed in ENUMS.items():
        if y.get(k) not in allowed:
            v.append(k)
    a = y.get('amount_paise')
    if a is not None and type(a) is not int:
        v.append('amount_type')
    d = y.get('txn_date')
    if d is not None and not (isinstance(d, str) and re.fullmatch(r'\d{4}-\d{2}-\d{2}', d)):
        v.append('date_format')
    t = y.get('txn_id')
    if t is not None and not (isinstance(t, str) and re.fullmatch(r'NP\d{10}', t)):
        v.append('txn_id_format')
    if type(y.get('needs_human')) is not bool:
        v.append('needs_human_type')
    return v

LEGAL = re.compile(r'\b(rbi|ombudsman|consumer\s+court|lawyer|legal\s+(notice|action))\b', re.I)
PRIOR_EXPLICIT = re.compile(
    r"(third|second|fourth|fifth|\d+(st|nd|rd|th)?) time I|again and again|"
    r"contacted (support|you) about this before|already (written|contacted|complained|raised|reported)|"
    r"written \d+ times|second complaint|keep raising|"
    r"baar baar|pehle bhi|teesri baar|dusri baar|doosri baar|kitni baar|phir se complaint", re.I)

def rule_priority(y, body):
    """SCHEMA.md v2 priority, applied in order (legal threats only in the customer's own text)."""
    c, a = y['category'], y['amount_paise']
    a = a if type(a) is int else None
    if c == 'fraud' or (a is not None and a >= 50_000_00) or LEGAL.search(first_part(body)):
        return 'P1'
    if (c in ('payment_failed', 'refund') and a is not None and a >= 5_000_00) or c == 'account_access':
        return 'P2'
    return 'P3'

# --- amounts in the text ---
NUM = r'\d[\d,]*(?:\.\d+)?'
MONEY_PATS = [
    re.compile(rf'(?:₹|\brs\.?|\binr)\s*(?P<n>{NUM})\s*(?P<m>k|lakhs?|lacs?)?\b', re.I),
    re.compile(rf'(?P<n>{NUM})\s*(?P<m>k|lakhs?|lacs?)\b', re.I),
    re.compile(rf'(?P<n>{NUM})\s*(?:rupees|rupee|rupaye|rupay)\b', re.I),
]
DISTRACT = re.compile(r'balance|limit|\bfees?\b|pade hain|bacha|remaining', re.I)

def to_paise(n, m=None):
    v = float(n.replace(',', ''))
    if m:
        v *= 1_000 if m.lower() == 'k' else 1_00_000
    return round(v * 100)

def boundaries(body):
    b = [-1]
    for m in re.finditer(r'[.!?](?=\s)|\n', body):
        if body[max(0, m.start() - 2):m.start()].lower() == 'rs':      # "Rs. 500" is not a sentence end
            continue
        b.append(m.start())
    b.append(len(body))
    return b

def sentence_of(body, s, e):
    b = boundaries(body)
    lo = max(x for x in b if x < s)
    hi = min(x for x in b if x >= e)
    return body[lo + 1:hi + 1].strip()

def money_mentions2(body):
    out = []
    for p in MONEY_PATS:
        for mt in p.finditer(body):
            try:
                paise = to_paise(mt.group('n'), mt.groupdict().get('m'))
            except ValueError:
                continue
            if any(o['paise'] == paise and abs(o['start'] - mt.start()) < 12 for o in out):
                continue
            sent = sentence_of(body, mt.start(), mt.end())
            out.append(dict(paise=paise, start=mt.start(), sent=sent, distractor=bool(DISTRACT.search(sent))))
    return out

def amount_status2(y, body):
    a, ms = y['amount_paise'], money_mentions2(body)
    real = [m for m in ms if not m['distractor']]
    if a is None:
        if not ms:
            return 'null & no money'
        return 'null, only distractors' if not real else 'NULL BUT REAL AMOUNT'
    hit = [m for m in ms if m['paise'] == a]
    if hit:
        return 'match' if any(not m['distractor'] for m in hit) else 'TOOK DISTRACTOR'
    if any(a * 100 == m['paise'] or a == m['paise'] // 100 for m in ms):
        return 'RUPEES NOT CONVERTED'
    return 'NOT FOUND'

def check_rows(row_ids, lab, body):
    """The data-audit validation gate: format, priority rule, needs_human rule, null txn fields, amount grounding."""
    problems = collections.Counter()
    for i in row_ids:
        y, b = lab[i], body[i]
        if violations(y):
            problems['format'] += 1
        if y['priority'] != rule_priority(y, b):
            problems['priority_rule'] += 1
        if y['needs_human'] != (y['priority'] == 'P1' or bool(PRIOR_EXPLICIT.search(first_part(b)))):
            problems['needs_human_rule'] += 1
        if y['category'] in NON_TXN and any(y[k] is not None for k in TXN_F):
            problems['txn_fields_not_null'] += 1
        if y['category'] in TXN_CATS and amount_status2(y, b) in ('NOT FOUND', 'RUPEES NOT CONVERTED', 'NULL BUT REAL AMOUNT'):
            problems['amount'] += 1
    return problems

p = check_rows(keep_ids, LAB, body_of)
print('clean rows re-checked with these rules:', len(keep_ids), '| problems:', dict(p) or 'NONE')
assert not p, 'the ported rules do not reproduce the cleaned labels: stop and investigate'

clean rows re-checked with these rules: 1262 | problems: NONE


## 2. Amount-boundary examples

A ticket is eligible when:
- it is a `payment_failed` or `refund` ticket with an amount, so the ₹5,000 rule applies;
- it has no legal threat, so its priority is decided by the amount alone;
- its label amount appears exactly once in the text as a non-distractor mention, so replacing it changes the claim
  and nothing else (amounts written in lakhs are skipped).

For each of the 250 sampled tickets, the threshold (₹5,000 or ₹50,000) is chosen at random and the new amount is
drawn around it, so examples fall on both sides of each boundary.

In [ ]:
rng = random.Random(1234)
aug, aug_log, aug_lab, aug_body = [], [], {}, {}

def indian(n):                                     # 168787 -> '1,68,787'
    s = str(n)
    if len(s) <= 3:
        return s
    head, tail, parts = s[:-3], s[-3:], []
    while len(head) > 2:
        parts.insert(0, head[-2:]); head = head[:-2]
    return ','.join(([head] if head else []) + parts + [tail])

def find_amount_match(body, paise):
    """The ONE non-distractor mention whose value equals the label amount (None if ambiguous)."""
    hits = []
    for p in MONEY_PATS:
        for mt in p.finditer(body):
            try:
                v = to_paise(mt.group('n'), mt.groupdict().get('m'))
            except ValueError:
                continue
            if v == paise and not DISTRACT.search(sentence_of(body, mt.start(), mt.end())):
                hits.append(mt)
    spans = {mt.span('n') for mt in hits}
    return sorted(hits, key=lambda mt: mt.start('n'))[0] if len(spans) == 1 else None

def sample_rupees(T):
    if rng.random() < 0.10:
        return T                                                     # exactly on the threshold ("or more")
    return int(round(math.exp(rng.uniform(math.log(0.4 * T), math.log(2.4 * T))), -2))

def render(old_n, mult, rupees):
    """Write `rupees` in the same style as the original number."""
    if mult and mult.lower() == 'k':
        return f'{rupees/1000:.1f}'.rstrip('0').rstrip('.')
    s = indian(rupees) if ',' in old_n else str(rupees)
    return s + (f'.{rng.randint(1, 99):02d}' if '.' in old_n else '')

def relabel(y, new_body, **changes):
    y2 = dict(y); y2.update(changes)
    y2['priority'] = rule_priority(y2, new_body)
    y2['needs_human'] = y2['priority'] == 'P1' or bool(PRIOR_EXPLICIT.search(first_part(new_body)))
    return {k: y2[k] for k in KEYS}

def add_aug(src, kind, new_body, y_new, extra):
    aid = f'aug-{kind}-{len(aug) + 1:04d}'
    user = user_of[src][:len(user_of[src]) - len(body_of[src])] + new_body      # same header, new body
    aug.append({'id': aid, 'messages': [
        {'role': 'system', 'content': SYS_PROMPT}, {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': json.dumps(y_new, ensure_ascii=False)}]})
    aug_lab[aid], aug_body[aid] = y_new, new_body
    aug_log.append({'id': aid, 'source': src, 'kind': kind, **extra})

amt_cands = []
for i in keep_ids:
    y, b = LAB[i], body_of[i]
    if y['category'] not in ('payment_failed', 'refund') or y['amount_paise'] is None:
        continue
    if LEGAL.search(first_part(b)):
        continue                                                     # keep priority driven by the amount alone
    mt = find_amount_match(b, y['amount_paise'])
    if mt is None or (mt.groupdict().get('m') or '').lower().startswith('l'):
        continue                                                     # ambiguous mention, or lakh-styled amount
    amt_cands.append((i, mt))
print('amount candidates:', len(amt_cands))

for i, mt in rng.sample(amt_cands, min(250, len(amt_cands))):
    b, y = body_of[i], LAB[i]
    mult = mt.groupdict().get('m')
    new_n = render(mt.group('n'), mult, sample_rupees(rng.choice([5_000, 50_000])))
    s, e = mt.span('n')
    nb = b[:s] + new_n + b[e:]
    y2 = relabel(y, nb, amount_paise=to_paise(new_n, mult))
    assert amount_status2(y2, nb) == 'match', (i, new_n)            # the new amount is grounded in the new text
    add_aug(i, 'amt', nb, y2, {'old_amt': y['amount_paise'], 'new_amt': y2['amount_paise'],
                               'pri': f"{y['priority']}->{y2['priority']}"})
print('amount examples made:', len(aug))

amount candidates: 531
amount examples made: 250


## 3. Legal-threat examples

SCHEMA.md makes any legal threat (RBI, ombudsman, consumer court, lawyer, legal notice) a P1 ticket. The threat
sentences come from the training tickets themselves, in the customer's own text. Only short, stand-alone sentences
are kept: no digits, no commas, no greeting. One is added to a ticket that is not fraud, not already P1 and has no
threat yet, in the ticket's language, just before the signature or quoted thread. Every result is asserted to
become P1 with `needs_human = true`. This teaches the model that the threat alone decides the priority.

In [ ]:
GREETINGS = ('hi', 'hello', 'hey', 'dear', 'namaste', 'bhai', 'sir', 'good morning', '@')
legal_sents = {'en': set(), 'hi-en': set()}
for i in keep_ids:
    fp = first_part(body_of[i])
    m = LEGAL.search(fp)
    if m:
        s = sentence_of(fp, m.start(), m.end())
        if (not re.search(r'\d', s) and len(s) < 80 and ',' not in s
                and not s.lower().startswith(GREETINGS)):              # a clean, stand-alone threat sentence
            legal_sents[LAB[i]['language']].add(s)
legal_sents = {k: sorted(v) for k, v in legal_sents.items()}
for k, v in legal_sents.items():
    print(f'{k}: {len(v)} sentences'); [print('   ', s) for s in v]

def insert_sentence(body, sent):
    cut = len(body)
    for marker in ('\n\nRegards', '\n\n>'):                          # before the signature / quoted thread
        k = body.find(marker)
        if k != -1:
            cut = min(cut, k)
    head = body[:cut].rstrip()
    return head + (' ' if head[-1:] in '.?!' else '. ') + sent + body[cut:]

leg_cands = [i for i in keep_ids if LAB[i]['category'] != 'fraud' and LAB[i]['priority'] != 'P1'
             and not LEGAL.search(first_part(body_of[i]))]
n0 = len(aug)
for i in rng.sample(leg_cands, min(120, len(leg_cands))):
    y = LAB[i]
    nb = insert_sentence(body_of[i], rng.choice(legal_sents[y['language']] or legal_sents['en']))
    y2 = relabel(y, nb)
    assert y2['priority'] == 'P1' and y2['needs_human']
    add_aug(i, 'legal', nb, y2, {'pri': f"{y['priority']}->P1"})
print('legal examples made:', len(aug) - n0)

en: 5 sentences
    I will complain to the RBI ombudsman.
    I will escalate this to the RBI.
    I will go to consumer court.
    My lawyer will send a legal notice.
    Otherwise I will approach the consumer court.
hi-en: 2 sentences
    Consumer court jaunga.
    RBI mein complaint kar dunga.
legal examples made: 120


## 4. Validation, examples and export

Every augmented row is re-checked against all SCHEMA.md rules. The priority transitions show the examples cover
the boundaries in both directions (for example P3→P2 and P2→P3), not just upgrades.

In [ ]:
p = check_rows(list(aug_lab), aug_lab, aug_body)
print('augmented rows checked:', len(aug_lab), '| problems:', dict(p) or 'NONE')
assert not p

al = pd.DataFrame(aug_log)
print('\nmade:', al['kind'].value_counts().to_dict())
print('priority transitions:\n', al.groupby('kind')['pri'].value_counts().to_string())
allp = pd.Series([y['priority'] for y in LAB.values()] + [y['priority'] for y in aug_lab.values()])
print('\npriority share  clean:', pd.Series([y['priority'] for y in LAB.values()]).value_counts(normalize=True).round(3).to_dict(),
      '| clean+aug:', allp.value_counts(normalize=True).round(3).to_dict())

for kind in ['amt', 'legal']:
    for _, a in al[al['kind'] == kind].head(4).iterrows():
        print('=' * 100)
        print(f"{a['id']}  (from {a['source']})")
        print('OLD:', repr(body_of[a['source']][:230]))
        print('    ', json.dumps(LAB[a['source']]))
        print('NEW:', repr(aug_body[a['id']][:230]))
        print('    ', json.dumps(aug_lab[a['id']]))

augmented rows checked: 370 | problems: NONE

made: {'amt': 250, 'legal': 120}
priority transitions:
 kind   pri   
amt    P3->P2    82
       P3->P1    38
       P2->P2    34
       P3->P3    29
       P2->P1    25
       P2->P3    15
       P1->P2    13
       P1->P1    11
       P1->P3     3
legal  P3->P1    82
       P2->P1    38

priority share  clean: {'P3': 0.561, 'P2': 0.246, 'P1': 0.193} | clean+aug: {'P3': 0.463, 'P2': 0.27, 'P1': 0.268}
aug-amt-0001  (from tr-01220)
OLD: 'Hello ji, Shopipng ka payment decline ho gaya lekin bank ne piase kaat liye. Rakam INR 43,984 hai. Account mein abhi 114 rupaye pade hain. Netbanking se hua tha. Jaldi solve karo please.'
     {"category": "payment_failed", "priority": "P2", "amount_paise": 4398400, "txn_date": null, "txn_id": null, "channel": "netbanking", "language": "hi-en", "needs_human": false}
NEW: 'Hello ji, Shopipng ka payment decline ho gaya lekin bank ne piase kaat liye. Rakam INR 61,700 hai. Account mein abhi 114 rupaye pade hain

In [ ]:
with open(f'{PREP}/train_aug.jsonl', 'w', encoding='utf-8') as f:
    for r in train + aug:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
al.to_csv(f'{PREP}/augmentation_log.csv', index=False)
print(f'wrote {PREP}/train_aug.jsonl: {len(train)} clean + {len(aug)} augmented = {len(train) + len(aug)} rows')
print(f'wrote {PREP}/augmentation_log.csv')

wrote /content/drive/MyDrive/vima3ya/prep_outputs/train_aug.jsonl: 1262 clean + 370 augmented = 1632 rows
wrote /content/drive/MyDrive/vima3ya/prep_outputs/augmentation_log.csv


## Result

The augmentation shifts the priority mix of the training set:

| Priority | Clean | Clean + augmented |
|---|---|---|
| P1 | 19.3% | 26.8% |
| P2 | 24.6% | 27.0% |
| P3 | 56.1% | 46.3% |

Dev has 21% P1, 26% P2 and 53% P3, so the augmented mix leans towards P1 relative to dev. Because the model has to
learn rules rather than base rates, I accepted this; dev exact match is the check.

Trained with the reference configuration (experiment D4), this data gave the best dev score: **95.0% exact match
against 91.5% for R1** (paired McNemar test: 8 tickets fixed, 1 broken, p = 0.039). Two retrains of the same
configuration scored 93.5% and 94.0%, so part of that margin is run-to-run noise. The gain over R1 held in all
three runs, but it is small.